# 01 — Treino do ResNet-50 (passo a passo)

Fine-tuning do **ResNet-50** (pre-treinado na ImageNet) para classificar **6 classes** de folhas de soja (5 doencas + saudavel), conforme o ADR-0001.

**Estrategia:** *two-phase fine-tuning* — primeiro com o backbone congelado (so o head aprende), depois descongelando tudo para ajuste fino, com *warmup* + *cosine decay* no learning rate.

**Pre-requisito:** rode o notebook `00_dataset_preparation` antes, para gerar os CSVs (`train.csv`/`val.csv`/`test.csv`/`label_map.csv`) em `data/processed/`.

**Ao final voce tera:** o melhor checkpoint, metricas (top-1, F1 macro), matriz de confusao e relatorio por classe — prontos para a apresentacao.

## 0. Ambiente (rode no Colab com GPU)

In [ ]:
import sys
from pathlib import Path

# Em Colab: instala dependencias, clona o repo e (opcional) monta o Drive
if 'google.colab' in sys.modules:
    !pip install -q timm albumentations scikit-learn pandas matplotlib seaborn tensorboard tqdm pyyaml onnx onnxruntime
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    from google.colab import drive
    drive.mount('/content/drive')

print('Setup OK')

## 1. Imports e configuracao

In [ ]:
import sys, csv
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('.').resolve()))

from src.utils.config import load_model_config
from src.utils.seed import set_seed
from src.data.transforms import get_train_transforms, get_val_transforms
from src.data.dataset import create_dataloaders
from src.models.factory import build_model, freeze_backbone, unfreeze_backbone
from src.training.losses import compute_class_weights, build_loss
from src.training.optim import build_optimizer, build_warmup_cosine_scheduler
from src.training.trainer import Trainer
from src.evaluation.evaluator import evaluate, save_metrics
from src.evaluation.confusion import plot_confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Carrega configs/base.yaml + configs/resnet50.yaml (override) e fixa a seed
cfg = load_model_config(Path('configs/resnet50.yaml'), Path('configs/base.yaml'))
set_seed(cfg['seed'])

model_name  = cfg['model']['name']
input_size  = cfg['model']['input_size']
num_classes = cfg['num_classes']

# Onde estao os CSVs gerados pelo notebook 00. No Colab, ajuste para o seu Drive,
# ex.: Path('/content/drive/MyDrive/ze-praga-dataset/processed')
DATA_DIR = Path('data/processed')

print('Modelo:', model_name, '| input:', input_size, '| classes:', num_classes)
print('Epocas:', cfg['epochs_total'], '| warmup:', cfg['epochs_warmup'], '| batch:', cfg['batch_size'])

## 2. Dados

Carregamos os *DataLoaders* a partir dos CSVs. O treino usa *augmentation* (flips, rotacao, brilho/contraste, blur, dropout); val/test usam apenas resize + normalizacao ImageNet.

In [ ]:
train_tf = get_train_transforms(input_size)
val_tf   = get_val_transforms(input_size)

train_loader, val_loader, test_loader = create_dataloaders(
    processed_dir=DATA_DIR,
    train_transform=train_tf,
    val_transform=val_tf,
    batch_size=cfg['batch_size'],
    num_workers=cfg['num_workers'],
)

with open(DATA_DIR / 'label_map.csv') as f:
    rows = sorted(csv.DictReader(f), key=lambda r: int(r['label_idx']))
    label_names = [r['label'] for r in rows]
print('Classes:', label_names)

## 3. Modelo (ResNet-50 via timm)

Carregamos o backbone pre-treinado e trocamos o *head* para 6 saidas. *Transfer learning* reaproveita as features aprendidas na ImageNet.

In [ ]:
model = build_model(model_name, num_classes=num_classes, pretrained=True)
n_params = sum(p.numel() for p in model.parameters())
print(model_name, '->', format(n_params, ','), 'parametros')

## 4. Loss, otimizador e scheduler

- **Loss:** CrossEntropy com *label smoothing* e **pesos de classe** (lidam com desbalanceamento entre as doencas).
- **Otimizador:** AdamW com *learning rates* separados (backbone < head).
- **Scheduler:** *warmup* linear seguido de *cosine decay*.

In [ ]:
class_weights = compute_class_weights(DATA_DIR / 'train.csv', num_classes)
criterion = build_loss(
    label_smoothing=cfg['loss']['label_smoothing'],
    class_weights=class_weights,
    device=DEVICE,
)

optimizer = build_optimizer(
    model, lr_backbone=3e-5, lr_head=3e-4,
    weight_decay=cfg['optimizer']['weight_decay'],
)
steps_per_epoch = len(train_loader)
total_steps  = cfg['epochs_total'] * steps_per_epoch
warmup_steps = int(cfg['scheduler']['warmup_steps_ratio'] * total_steps)
scheduler = build_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

print('Pesos de classe:', [round(w, 3) for w in class_weights])
print('Steps totais:', total_steps, '| warmup:', warmup_steps)

## 5. Treinamento

O `Trainer` faz o *two-phase fine-tuning* com *mixed precision* (AMP), *gradient clipping*, checkpoint do melhor modelo (por `val_f1_macro`) e *early stopping*. As curvas vao para o TensorBoard.

In [ ]:
# (Opcional) acompanhe as curvas em tempo real
%load_ext tensorboard
%tensorboard --logdir artifacts/tensorboard

In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=DEVICE,
    model_name=model_name,
    mixed_precision=cfg.get('mixed_precision', True),
    gradient_clip_norm=cfg.get('gradient_clip_norm', 1.0),
    tensorboard_dir=cfg['logging']['tensorboard_dir'],
    log_every_n_steps=cfg['logging']['log_every_n_steps'],
)

best_model = trainer.fit(
    epochs_total=cfg['epochs_total'],
    epochs_warmup=cfg['epochs_warmup'],
    patience=cfg['patience_early_stop'],
    freeze_fn=freeze_backbone,
    unfreeze_fn=unfreeze_backbone,
)
print('Treino concluido.')

## 6. Avaliacao no test set

In [ ]:
results = evaluate(best_model, test_loader, label_names, device=DEVICE)
print('Acuracia (top-1):', round(results['accuracy'], 4))
print('F1 macro:        ', round(results['f1_macro'], 4))
print('F1 weighted:     ', round(results['f1_weighted'], 4))

save_metrics(results, Path('artifacts/metrics') / ('metrics_' + model_name + '.json'), model_name)

## 7. Matriz de confusao

In [ ]:
fig = plot_confusion_matrix(
    results['y_true'], results['y_pred'], label_names,
    save_path=Path('artifacts/figures') / ('confusion_' + model_name + '.png'),
    title='Matriz de confusao - ' + model_name,
)
plt.show()

## 8. Relatorio por classe

Precisao, recall e F1 por classe — util para ver onde o modelo confunde (ex.: cercosporiose vs. mancha-alvo).

In [ ]:
report = pd.DataFrame(results['per_class']).T
report

## 9. Checkpoint salvo

In [ ]:
ckpt = Path('artifacts/checkpoints') / ('best_' + model_name + '.pth')
print('Melhor checkpoint (por val_f1_macro):', ckpt)
print('Existe?', ckpt.exists())
print('Proximo passo: exportar para ONNX (notebook 04_export_onnx).')